In [1]:
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime import QiskitRuntimeService
import qiskit_ibm_runtime
from qiskit import transpile
from qiskit.visualization import plot_histogram
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.primitives import BackendSamplerV2
from qiskit_ibm_runtime import SamplerV2 as Sampler
import numpy as np
from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister

In [2]:
import qiskit
import qiskit_ibm_runtime

print(qiskit.__version__)
print(qiskit_ibm_runtime.__version__)

2.3.0
0.45.0


In [3]:
bit_num = 127
qr = QuantumRegister(bit_num, "q")
cr = ClassicalRegister(bit_num, "c")
qc = QuantumCircuit(qr, cr)
rng = np.random.default_rng()

In [4]:
abits = np.round(rng.random(bit_num))
abase = np.round(rng.random(bit_num))

for i in range(bit_num):
    if abits[i] == 0:
        if abase[i] ==1:
            qc.h(i)
    if abits[i] == 1:
        if abase[i] == 1:
            qc.x(i)
        if abase[i] == 0:
            qc.x(i)
            qc.h(i)

### Evesdropper comes into the game

In [5]:
#Evesdropper will draw some random bases 
ebase = np.round(rng.random(bit_num))

for k in range(bit_num):
    if ebase[k] == 1:
        qc.h(k)
    qc.measure(qr[k], cr[k])

In [6]:
print(qiskit_ibm_runtime.__version__)

0.45.0


In [7]:
QiskitRuntimeService.delete_account()

False

In [8]:
QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token="m5VxlOJ23TacrgHkR0lzMxnofjDNs-jd-4tdIEfxpRKz",
    overwrite=True
)

In [9]:
service = QiskitRuntimeService(channel="ibm_quantum_platform")

print(service.backends())

IBMInputValueError: 'No matching instances found for the following filters: .'

In [ ]:
backend = service.backend("ibm_brisbane")
print(backend.name)
print("Backend:", backend)
print("Backend type:", type(backend))

In [ ]:
#Now transpiling
target = backend.target
pm = generate_preset_pass_manager(target=target, optimization_level=3)
qc_isa = pm.run(qc)

In [ ]:
sampler = Sampler(mode=backend)

In [ ]:
job = sampler.run([qc_isa], shots=1)
counts = job.result()[0].data.c.get_counts()
countsint = job.result()[0].data.c.get_int_counts()

In [ ]:
#Post-processing
#Now extracting Eve's bit
keys = counts.keys()
key = list(keys)[0]
emeas = list(key)
emeas_ints = []
for n in range(bit_num):
    emeas_ints.append(int(emeas[n]))
#Reversing the results for bits
ebits = emeas_ints[::-1]
print(ebits)

### Evesdropper uses her measurements above to prepare best guess to send on to bob

In [ ]:
qr = QuantumRegister(bit_num, "q")
cr = ClassicalRegister(bit_num, "c")
qc = QuantumCircuit(qr, cr)

In [ ]:
#State preparation by eve
for p in range(bit_num):
    if ebits[p] == 0:
        if ebase[p] == 1:
            qc.h(p)
    if ebits[p] == 1:
        if ebase[p] == 0:
            qc.x(p)
        if ebase[p] == 1:
            qc.x(p)
            qc.h(p)

In [ ]:
#Now Bob will have his random bases
bbase = np.round(rng.random(bit_num))

for m in range(bit_num):
    if bbase[m] == 1:
        qc.h(m)
    qc.measure(qr[m], cr[m])

In [ ]:
#Transpiling
target = backend.target
pm = generate_preset_pass_manager(target=target, optimization_level=3)
qc_isa = pm.run(qc)

In [ ]:
#Generating the bob bits
job = sampler.run([qc_isa], shots=1)
counts = job.result()[0].data.c.get_counts()
countsint = job.result()[0].data.c.get_int_counts()

In [ ]:
#post-processing for Bob
#extracting Bob's bits
keys = counts.keys()
key = list(keys)[0]
bmeas = list(key)
bmeas_ints = []
for y in range(bit_num):
    bmeas_ints.append(int(bmeas[y]))
#Rversing 
bbits = bmeas_ints[::-1]

### Now Bits Comparision between Alice and Bob,

In [ ]:
#When only bits are same, keeping the bits
agoodbits = []
bgoodbits = []
match_count = 0
for z in range(bit_num):
    if abase[z] == bbase[z]:
        agoodbits.append(int(abits[z]))
        bgoodbits.append(bbits[z])
        if int(abits[z]) == bbits[z]:
            match_count += 1

In [ ]:
#Now printing the results
print("Alice's bits = ", agoodbits)
print("Bob's bits = ", bgoodbits)
print("fidelity = ", match_count / len(agoodbits))
print("loss = ", 1 - match_count / len(agoodbits))